# Notebook 2 — LangGraph Basics
### What is LangGraph?
LangGraph is a way to build **workflows** where each step is an AI-powered function.

If LangChain is the individual worker, LangGraph is the **org chart** that defines:
- Who does what (nodes)
- What order they do it (edges)
- What information they share (state)

Think of it like a flowchart that executes automatically.

**By the end of this notebook you will know:**
1. What State is and why it matters
2. How to write a Node
3. How to connect Nodes with Edges
4. How to build and run a Graph
5. How to add conditional branching (if/else in the flow)

> Pre-requisite: Complete Notebook 1 first

---
## Step 1 — Install

In [1]:
# langgraph — the workflow orchestration library
%pip install -q langchain langchain-ollama langgraph

Note: you may need to restart the kernel to use updated packages.


---
## Step 2 — Imports and LLM Setup

In [1]:
# TypedDict lets us define a typed dictionary — used for State
from typing import TypedDict

# StateGraph — the main class to build a graph
# END        — a special marker that means 'stop the graph here'
from langgraph.graph import StateGraph, END

# LangChain components from Notebook 1
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print('Imports OK')

Imports OK


In [2]:
# Create the LLM — same as Notebook 1
llm = ChatOllama(
    model='llama3.2',
    base_url='http://localhost:11434',
    temperature=0.3,
)
print('LLM ready')

LLM ready


---
## Step 3 — Understanding State
**State** is a shared dictionary that every node can read from and write to.
Think of it as a **notepad** passed around between workers.

- Node 1 reads the notepad, does its job, writes its result back
- Node 2 picks up the notepad, reads Node 1's result, adds its own
- And so on until the end

We define State as a `TypedDict` — this tells Python what fields exist and their types.

In [3]:
# This is our State — a typed dictionary with 3 fields
# Each field represents one piece of information flowing through the graph
class SimpleState(TypedDict):
    input_text: str    # the original input — written once at the start
    step1_result: str  # written by Node 1, read by Node 2
    step2_result: str  # written by Node 2 — final output

print('State defined')
print('Fields:', list(SimpleState.__annotations__.keys()))

State defined
Fields: ['input_text', 'step1_result', 'step2_result']


---
## Step 4 — Writing Nodes
A **Node** is just a Python function that:
1. Receives the full State dictionary
2. Does some work (usually calls the LLM)
3. Returns an updated State dictionary

Important rule: always return `{**state, 'field': new_value}`
The `**state` keeps all existing fields — you only update what you change.

In [4]:
# Node 1 — Summariser
# Job: read input_text, produce a short summary, write to step1_result

summarise_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a summariser. Summarise the input in exactly 2 sentences.'),
    ('human',  '{text}')
])
summarise_chain = summarise_prompt | llm | StrOutputParser()

def node_summariser(state: SimpleState) -> SimpleState:
    print('[Node 1 - Summariser] Running...')

    # Read from state — get the original input
    text = state['input_text']

    # Do the work — call the LLM chain
    summary = summarise_chain.invoke({'text': text})

    # Return updated state — keep everything, add step1_result
    return {**state, 'step1_result': summary}

print('Node 1 defined')

Node 1 defined


In [5]:
# Node 2 — Question Generator
# Job: read the summary from step1_result, generate 3 test questions

question_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a QA analyst. Generate exactly 3 test questions based on the input.'),
    ('human',  'Generate 3 test questions for this:\n\n{summary}')
])
question_chain = question_prompt | llm | StrOutputParser()

def node_question_generator(state: SimpleState) -> SimpleState:
    print('[Node 2 - Question Generator] Running...')

    # Read from state — get Node 1's output
    summary = state['step1_result']

    # Do the work
    questions = question_chain.invoke({'summary': summary})

    # Return updated state — keep everything, add step2_result
    return {**state, 'step2_result': questions}

print('Node 2 defined')

Node 2 defined


---
## Step 5 — Building the Graph
Now we wire everything together:
1. Create a `StateGraph` and tell it which State to use
2. Add each node with a name
3. Set the entry point (where the graph starts)
4. Add edges (arrows showing what runs next)
5. Compile the graph (makes it ready to run)

In [6]:
# Step 1: Create the graph, tell it what State type to use
graph = StateGraph(SimpleState)

# Step 2: Add nodes — give each node a name and point to the function
graph.add_node('summariser',         node_summariser)
graph.add_node('question_generator', node_question_generator)

# Step 3: Set the entry point — which node runs first
graph.set_entry_point('summariser')

# Step 4: Add edges — after 'summariser' runs, go to 'question_generator'
graph.add_edge('summariser', 'question_generator')

# After 'question_generator' runs, stop (END)
graph.add_edge('question_generator', END)

# Step 5: Compile the graph — makes it executable
app = graph.compile()

print('Graph compiled successfully')
print('Flow: summariser → question_generator → END')

Graph compiled successfully
Flow: summariser → question_generator → END


---
## Step 6 — Running the Graph
Call `.invoke()` with the **initial state** — only the fields you have at the start.
LangGraph fills in the rest as each node runs.

In [7]:
# Define the starting state
# Only input_text is filled — the other fields start empty
initial_state: SimpleState = {
    'input_text': (
        'A fund transfer feature allows bank customers to move money '
        'between their own accounts or to third-party accounts. '
        'Transfers are limited to $10,000 per day and require SMS OTP confirmation.'
    ),
    'step1_result': '',   # empty — Node 1 will fill this
    'step2_result': '',   # empty — Node 2 will fill this
}

print('Running graph...')
print('-' * 50)

# .invoke() runs the entire graph from entry point to END
result = app.invoke(initial_state)

print('-' * 50)
print('Graph completed')

Running graph...
--------------------------------------------------
[Node 1 - Summariser] Running...
[Node 2 - Question Generator] Running...
--------------------------------------------------
Graph completed


In [9]:
# Inspect each field in the result state
print('ORIGINAL INPUT:')
print(result['input_text'])
print()
print('NODE 1 OUTPUT (Summary):')
print(result['step1_result'])
print()
print('NODE 2 OUTPUT (Test Questions):')
print(result['step2_result'])

ORIGINAL INPUT:
A fund transfer feature allows bank customers to move money between their own accounts or to third-party accounts. Transfers are limited to $10,000 per day and require SMS OTP confirmation.

NODE 1 OUTPUT (Summary):
Bank customers can use a fund transfer feature to send money to either their own account or to another individual's account. The service has daily limits of $10,000 and requires an additional one-time SMS OTP confirmation for verification.

NODE 2 OUTPUT (Test Questions):
Here are three test questions based on the input:

1. What is the maximum amount that can be transferred using the fund transfer feature in a single transaction?

(Answer: $10,000)

2. Does the fund transfer feature require any additional verification steps beyond the daily limit, and if so, what is the nature of this verification?

(Answer: Yes, an additional one-time SMS OTP confirmation is required for verification.)

3. Can bank customers use the fund transfer feature to send money to a

---
## Step 7 — Conditional Edges (if/else in the graph)
Sometimes the next node depends on what the previous one found.
LangGraph supports this with **conditional edges** — a function decides where to go next.

Example: After analysing a test result, route to 'pass handler' or 'fail handler'.

In [10]:
# New State for this example
class ReviewState(TypedDict):
    test_result: str    # 'pass' or 'fail' — decides routing
    feedback:    str    # output from whichever branch runs


# Node A — checks the test result and sets the route
def node_checker(state: ReviewState) -> ReviewState:
    print(f'[Checker] Test result is: {state["test_result"]}')
    # This node just passes state through — the routing logic is separate
    return state


# Node B — runs when test passed
def node_pass_handler(state: ReviewState) -> ReviewState:
    print('[Pass Handler] Running...')
    return {**state, 'feedback': 'All tests passed. Ready for release review.'}


# Node C — runs when test failed
def node_fail_handler(state: ReviewState) -> ReviewState:
    print('[Fail Handler] Running...')
    return {**state, 'feedback': 'Tests failed. Defects must be fixed before release.'}


# The routing function — reads state and returns the NEXT NODE NAME as a string
def route_by_result(state: ReviewState) -> str:
    if state['test_result'].lower() == 'pass':
        return 'pass_handler'   # return the node name to go to
    else:
        return 'fail_handler'


print('Nodes and routing function defined')

Nodes and routing function defined


In [11]:
# Build the conditional graph
cond_graph = StateGraph(ReviewState)

cond_graph.add_node('checker',      node_checker)
cond_graph.add_node('pass_handler', node_pass_handler)
cond_graph.add_node('fail_handler', node_fail_handler)

cond_graph.set_entry_point('checker')

# add_conditional_edges — after 'checker', call route_by_result()
# The result of that function decides which node runs next
# The dict maps function return values to node names
cond_graph.add_conditional_edges(
    'checker',                          # from this node
    route_by_result,                    # call this routing function
    {
        'pass_handler': 'pass_handler', # if function returns 'pass_handler' → go there
        'fail_handler': 'fail_handler'  # if function returns 'fail_handler' → go there
    }
)

# Both branches end at END
cond_graph.add_edge('pass_handler', END)
cond_graph.add_edge('fail_handler', END)

cond_app = cond_graph.compile()
print('Conditional graph compiled')

Conditional graph compiled


In [12]:
# Test the PASS route
pass_result = cond_app.invoke({'test_result': 'pass', 'feedback': ''})
print('Feedback:', pass_result['feedback'])
print()

[Checker] Test result is: pass
[Pass Handler] Running...
Feedback: All tests passed. Ready for release review.



In [13]:
# Test the FAIL route
fail_result = cond_app.invoke({'test_result': 'fail', 'feedback': ''})
print('Feedback:', fail_result['feedback'])

[Checker] Test result is: fail
[Fail Handler] Running...
Feedback: Tests failed. Defects must be fixed before release.


---
## Summary — What You Learned

| Concept | What it is | Code |
|---|---|---|
| State | Shared notepad between nodes | `class MyState(TypedDict): field: str` |
| Node | A function that reads/writes state | `def my_node(state) -> state:` |
| StateGraph | The graph container | `StateGraph(MyState)` |
| add_node | Register a node | `graph.add_node('name', function)` |
| set_entry_point | Where the graph starts | `graph.set_entry_point('name')` |
| add_edge | Connect two nodes | `graph.add_edge('a', 'b')` |
| add_conditional_edges | If/else routing | `graph.add_conditional_edges(...)` |
| compile | Make graph runnable | `app = graph.compile()` |
| invoke | Run the graph | `app.invoke(initial_state)` |

**You are now ready for Notebook 3 — Combined QA Example.**